# Phase 6: Metrics Agent (Subscriber + Publisher)

This notebook subscribes to spectator events and publishes final KPI metrics at halftime end.

In [1]:
# Cell 2 purpose: Import modules and ensure src is available from notebooks/.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.metrics import (
    MetricsAggregatorState,
    enforce_final_scenario_policies,
    finalize_kpi_payload,
    record_spectator_event,
)
from simulated_city.topic_schema import topic_kpi_metrics, topic_spectator_events

In [5]:
# Cell 3 purpose: Load config, create metrics aggregator, and connect MQTT client.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

enforce_final_scenario_policies(
    missed_kickoff_timestamp_s=900,
    external_disruptions_enabled=False,
    group_coordination_share=0.2,
)
state = MetricsAggregatorState(halftime_duration_s=900)

spectator_topic = topic_spectator_events()
kpi_topic = topic_kpi_metrics()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='metrics-agent')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topic: {spectator_topic}')
print(f'Publish topic: {kpi_topic}')

Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/events/spectator
Publish topic: stadium/a4/halftime/metrics/kpi


In [6]:
# Cell 4 purpose: Subscribe to spectator events and feed metrics aggregator.
received_events = []
published_kpi_payloads = []

def _on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    received_events.append(payload)
    record_spectator_event(state, payload)

mqtt_client.on_message = _on_message
mqtt_client.subscribe(spectator_topic, qos=1)
print('Subscription started. Waiting for incoming spectator events...')

Subscription started. Waiting for incoming spectator events...


In [7]:
# Cell 5 purpose: Run short loop, publish final KPI snapshot, and disconnect.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

run_id = received_events[-1]['run_id'] if received_events else 'a4-no-events'
final_payload = finalize_kpi_payload(state=state, run_id=run_id, timestamp_s=900)
ok = mqtt.publish_json_checked(mqtt_client, kpi_topic, final_payload, qos=1)
if ok:
    published_kpi_payloads.append(final_payload)

print(f'Received spectator events: {len(received_events)}')
print(f'Published KPI payloads: {len(published_kpi_payloads)}')
if published_kpi_payloads:
    print('Final KPI payload:')
    print(published_kpi_payloads[-1])

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

Received spectator events: 1
Published KPI payloads: 1
Final KPI payload:
{'schema_version': '1.0', 'run_id': 'a4-run-21cbe7e8', 'timestamp_s': 900, 'average_wait_s': 0.0, 'wait_percentiles_s': {'P01': 0.0, 'P02': 0.0, 'P03': 0.0, 'P04': 0.0, 'P05': 0.0, 'P06': 0.0, 'P07': 0.0, 'P08': 0.0, 'P09': 0.0, 'P10': 0.0, 'P11': 0.0, 'P12': 0.0, 'P13': 0.0, 'P14': 0.0, 'P15': 0.0, 'P16': 0.0, 'P17': 0.0, 'P18': 0.0, 'P19': 0.0, 'P20': 0.0, 'P21': 0.0, 'P22': 0.0, 'P23': 0.0, 'P24': 0.0, 'P25': 0.0, 'P26': 0.0, 'P27': 0.0, 'P28': 0.0, 'P29': 0.0, 'P30': 0.0, 'P31': 0.0, 'P32': 0.0, 'P33': 0.0, 'P34': 0.0, 'P35': 0.0, 'P36': 0.0, 'P37': 0.0, 'P38': 0.0, 'P39': 0.0, 'P40': 0.0, 'P41': 0.0, 'P42': 0.0, 'P43': 0.0, 'P44': 0.0, 'P45': 0.0, 'P46': 0.0, 'P47': 0.0, 'P48': 0.0, 'P49': 0.0, 'P50': 0.0, 'P51': 0.0, 'P52': 0.0, 'P53': 0.0, 'P54': 0.0, 'P55': 0.0, 'P56': 0.0, 'P57': 0.0, 'P58': 0.0, 'P59': 0.0, 'P60': 0.0, 'P61': 0.0, 'P62': 0.0, 'P63': 0.0, 'P64': 0.0, 'P65': 0.0, 'P66': 0.0, 'P67': 0.0, '

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Disconnected from MQTT broker.
